# Brand value and the sponsorship's incremental brand value

**Review notebook** for `brand_value_dominance_v1` (brand share of share-price formation and the Index-lift chain; ADR-0031, ADR-0032, ADR-0040), with the comparison routes `sponsorship_total_return_v3` and `valuation_routes_v1` (ADR-0030). It follows the human-review standard in `AGENTS.md`. The economic-profit DCF (`brand_value_dcf_v1`, ADR-0033) was retired by the owner in ADR-0040 and is no longer computed.

> **Status: planning valuation, not accepted.** The brand-share input is an owner-selected proxy and the sponsorship lift it is applied to is borderline significant (see `10_total_sponsorship_effect.ipynb`). Quote any figure here only with its range and these caveats.

*How to use:* run all cells. Read-only unless `RUN_REBUILD = True`. Narrative numbers are computed from current outputs.

In [1]:
# Setup: read-only by default (AGENTS.md). Nothing below rewrites data unless RUN_REBUILD is True.
RUN_REBUILD = False   # set True only to regenerate the production outputs this notebook reads

import json, os, subprocess, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "srmp").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from srmp.review import stamp_sources

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (10, 4.2), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

def md(text):
    display(Markdown(text))

def fmt(value, digits=2, signed=False):
    return f"{value:+.{digits}f}" if signed else f"{value:.{digits}f}"

if RUN_REBUILD:  # explicit opt-in: re-runs the valuation experiments (about a minute)
    for module in ["srmp.experiments.stock_brand_response_v2", "srmp.experiments.sponsorship_total_return_v3",
                   "srmp.experiments.valuation_routes_v1", "srmp.experiments.brand_value_dominance_v1"]:
        subprocess.run([sys.executable, "-m", module], check=True)

E = "data/curated/experimental/"
D = E + "brand_value_dominance_v1/"
SOURCES = [
    "config/experiments/brand_value_dominance_v1.yaml", "data/curated/index/brand_index_manifest.json",
    E + "stock_brand_response_v2/stock_response_manifest.json",
    D + "brand_value_dominance_manifest.json", D + "dominance_weights.parquet", D + "stability.parquet",
    E + "sponsorship_total_return_v3/total_return_v3_manifest.json",
    E + "valuation_routes_v1/valuation_routes.parquet", E + "valuation_routes_v1/breakeven.parquet",
    E + "sponsorship_total_effect_v1/estimate_manifest.json", E + "sponsorship_total_effect_v1/donor_sensitivity.parquet",
    E + "sponsorship_exposure_timing_v1/exposure_timing_manifest.json",
]
stamp_sources(SOURCES)
dom = json.loads(Path(D + "brand_value_dominance_manifest.json").read_text())
values, chain = dom["values"], dom["primary_result"]
fs = dom["fifa_specific_incremental_brand_value"]
effect = json.loads(Path(E + "sponsorship_total_effect_v1/estimate_manifest.json").read_text())

REVIEW_SOURCES={"config/experiments/brand_value_dominance_v1.yaml": "0322a971cad4c7d3cba94ab093ae88afd74c8dc5ce739dc3a5c87ce5a423faa1", "data/curated/index/brand_index_manifest.json": "18db10277b376a95c97748dafcc8c1b384cbf991bab061cf190af77e0bb0051e", "data/curated/experimental/stock_brand_response_v2/stock_response_manifest.json": "292662068ea28ec8936c019ae1ac9581f8a4b49e9df1b80486b7db010d4de64b", "data/curated/experimental/brand_value_dominance_v1/brand_value_dominance_manifest.json": "43e9475712d03331423af51eb10a19dcb7094d3745dea785b767b425b45ff3f1", "data/curated/experimental/brand_value_dominance_v1/dominance_weights.parquet": "9680cc053917fb0d49b228484dbe40427a5ddf3be917ebfa22c9b9779297a770", "data/curated/experimental/brand_value_dominance_v1/stability.parquet": "84b090622cda57da524d5cd3ef1e60a248811032a4be078e1369ab0730c4c7b7", "data/curated/experimental/sponsorship_total_return_v3/total_return_v3_manifest.json": "7c02e6fdac827993171debe1e1bb2119cf6011ac9aa3412723190887b1ab328a

## 1. Question and purpose

**Questions.** (a) What is the Lenovo brand worth, read as the brand's share of what moves Lenovo's share price? (b) Given the sponsorship's measured lift in the Brand Index, how much incremental brand value does that lift imply?

**Why it matters.** The client needs a money translation of the sponsorship's brand effect. This notebook makes every step of that translation explicit, so the size of each assumption is visible.

**Objective supported.** Monetisation of the sponsorship's brand uplift (downstream of, and conditional on, the uplift analysis).

## 2. Evidence and inputs

| Input | Value / source | Label |
|---|---|---|
| Lenovo weekly log returns, Hang Seng and lagged Nasdaq-100 (HKD) returns | Yahoo Finance daily prices | **observed** |
| Brand-index innovation | Unexpected weekly change in the Brand Index, predicted from prior data only (`stock_brand_response_v2`) | **estimated** |
| Category demand, Lenovo events (earnings, launches, promotions) | Google Trends rival searches; Lenovo press and results calendars | **observed** / **constructed** |
| Market capitalisation | Mean over the post-announcement weeks (primary), latest and pre-announcement (sensitivities) | **observed** |
| Sponsorship lift (primary) | FIFA-specific effect: the part of the synthetic-control gap explained by accumulated FIFA exposure (ADR-0043) | **estimated** (regression decomposition) |
| Sponsorship lift (upper band) | Total sponsorship effect on the Brand Index (synthetic control) | **estimated**, borderline significant |
| Brand value elasticity to brand share = 1 | Proportionality | **assumed** |
| Brand Finance 2025 brand value | Brand Finance China 500 2025 | **borrowed** (external reference only, never an input) |

## 3. Method and formulas

**Brand share of price formation (dominance analysis).** Weekly Lenovo log returns $r_t$ are regressed on four groups of drivers $G$: market factors, category demand, Lenovo events, and the Brand Index innovation. General dominance gives each group its average incremental $R^2$ over all orderings of the other groups (Shapley–Owen):

$$D_g = \frac{1}{|G|}\sum_{k=0}^{|G|-1} \overline{\Big[R^2(S \cup \{g\}) - R^2(S)\Big]}_{|S|=k,\ g \notin S}, \qquad \sum_g D_g = R^2_{full}.$$

*In words:* how much of the explained movement in Lenovo's share price each group accounts for, fairly averaged over the order in which groups are added. The brand's share is signed by its full-model coefficient so that adverse co-movement is not counted as value:

$$s = \operatorname{sign}(\hat\beta_{brand}) \cdot \frac{D_{brand}}{R^2_{full}}, \qquad s_{L} = \operatorname{sign}(\hat\beta_{brand}) \cdot \frac{D_{brand}}{\sum_{g \ne market} D_g}.$$

$s$ is the owner-selected literal reading (share of all explained movement); $s_L$ the Lenovo-specific reading (share of the non-market drivers).

**Brand value.** $BV = s \cdot \text{MarketCap}$ (and $BV_L = s_L \cdot \text{MarketCap}$). *In words:* the brand is worth the share of the company's value that corresponds to its share of price formation (ADR-0031).

**From Index lift to incremental brand value.** The Brand Index is proportional to Lenovo's brand share of attention (zero share = 0; ADR-0040). A lift $\Delta$ over the no-sponsorship level $I_{cf}$ is therefore an uplift

$$u = \frac{\Delta}{I_{cf}}, \qquad \Delta BV = \varepsilon \cdot u \cdot BV, \qquad \text{coefficient} = \frac{\varepsilon\, BV}{I_{cf}}\ \text{per Index point}.$$

*In words:* the lift is converted into "percent more brand share of attention", and brand value is assumed to move one-for-one ($\varepsilon = 1$) with it.

## 4. Calculation path

| Step | Code | Configuration | Output |
|---|---|---|---|
| Brand Index | `srmp/index/brand_factor.py` | `config/brand_index.yaml` | `data/curated/index/brand_index_weekly.parquet` |
| Stock-return drivers, brand innovation | `srmp/experiments/stock_brand_response_v2.py` | `config/experiments/stock_brand_response_v2.yaml` | `stock_brand_response_v2/` |
| Total sponsorship effect (the lift) | `srmp/experiments/sponsorship_total_effect_v1.py` | `config/experiments/sponsorship_total_effect_v1.yaml` | `sponsorship_total_effect_v1/` |
| Market capitalisation bases, sustained level (v3) | `srmp/experiments/sponsorship_total_return_v3.py` | `config/experiments/sponsorship_total_return_v3.yaml` | `sponsorship_total_return_v3/` |
| Dominance shares, brand value, Index-lift chain | `srmp/experiments/brand_value_dominance_v1.py` (`group_dominance`, `_ratio_scale`) | `config/experiments/brand_value_dominance_v1.yaml` | `brand_value_dominance_v1/` |
| Comparison routes and breakevens | `srmp/experiments/valuation_routes_v1.py` | `config/experiments/valuation_routes_v1.yaml` | `valuation_routes_v1/` |

## 5. Results

### 5.1 What moves Lenovo's share price

In [2]:
wts = pd.read_parquet(D + "dominance_weights.parquet")
display(wts.rename(columns={"group": "Driver group", "dominance_r2": "Dominance (R² points)", "share_of_explained": "Share of explained movement",
                            "signed_share_of_explained": "Signed share"}).drop(columns="model")
        .style.format({"Dominance (R² points)": "{:.4f}", "Share of explained movement": "{:.1%}", "Signed share": "{:.2%}"}, na_rep="—").hide(axis="index"))
coef = dom["coefficients"]["brand_full_sample"]["brand"]
md(f"**Table 1.** {dom['samples']['brand_weeks']} weeks; full-model R² = {dom['samples']['brand_model_r2']:.3f}. "
   f"Brand innovation coefficient {coef['coefficient']:+.5f} (robust s.e. {coef['robust_se']:.5f}, t = {coef['coefficient']/coef['robust_se']:+.2f}). "
   "Market factors explain most weekly movement; the brand group explains little.")

Driver group,Dominance (R² points),Share of explained movement,Signed share
market,0.1640,91.2%,—
category_demand,0.0042,2.3%,-2.34%
lenovo_events,0.0079,4.4%,—
brand,0.0038,2.1%,2.10%


**Table 1.** 164 weeks; full-model R² = 0.180. Brand innovation coefficient +0.00298 (robust s.e. 0.00255, t = +1.17). Market factors explain most weekly movement; the brand group explains little.

In [3]:
stab = pd.read_parquet(D + "stability.parquet")
display(stab.rename(columns={"check": "Stability check", "weeks": "Weeks", "r2": "Model R²", "brand_share_of_explained": "Brand share (literal)",
                             "brand_value_usd_m": "Brand value (US$m)", "incremental_brand_value_usd_m": "Incremental BV (US$m)"})
        .style.format({"Model R²": "{:.3f}", "Brand share (literal)": "{:.2%}", "Brand value (US$m)": "{:,.0f}",
                       "Incremental BV (US$m)": "{:,.1f}"}).hide(axis="index"))
md(f"**Table 2.** The literal brand share ranges {100*dom['stability_summary']['brand_share_min']:.2f}% to {100*dom['stability_summary']['brand_share_max']:.2f}% "
   "across sample windows, control groups and brand measures: share-price data pin the brand's role down only loosely.")

Stability check,Weeks,Model R²,Brand share (literal),Brand value (US$m),Incremental BV (US$m)
sample 2022-01-01 to 2024-10-14,62,0.363,0.66%,140,10.3
sample 2024-10-15 to 2026-12-31,102,0.167,4.11%,872,64.0
sample 2022-01-01 to 2025-12-31,125,0.360,2.54%,539,39.5
without market,164,0.013,21.09%,"4,473",328.0
without category_demand,164,0.176,1.67%,354,25.9
without lenovo_events,164,0.167,1.63%,345,25.3
brand measured as delta_brand_index,164,0.178,1.91%,406,29.8


**Table 2.** The literal brand share ranges 0.66% to 21.09% across sample windows, control groups and brand measures: share-price data pin the brand's role down only loosely.

### 5.2 Brand value

In [4]:
ref = dom["external_reference"]
boot = values["brand_value_90pct_usd_m"]
bv = pd.DataFrame([
    {"Reading": "Literal: share of all explained movement (owner-selected primary)", "Brand share": values["brand_share_of_price_formation"], "Brand value (US$m)": values["brand_value_usd_m"]},
    {"Reading": "Lenovo-specific: share of the non-market drivers", "Brand share": values["brand_share_of_lenovo_specific_formation"], "Brand value (US$m)": values["brand_value_lenovo_specific_usd_m"]},
    {"Reading": "Lower bound: share of total return variance", "Brand share": values["brand_share_of_total_variance"], "Brand value (US$m)": values["brand_value_lower_bound_usd_m"]},
    {"Reading": "Brand Finance 2025 (external, royalty-relief method)", "Brand share": ref["brand_finance_2025_brand_value_usd_m"] / values["market_cap_usd_m"], "Brand value (US$m)": ref["brand_finance_2025_brand_value_usd_m"]},
])
display(bv.style.format({"Brand share": "{:.2%}", "Brand value (US$m)": "{:,.0f}"}).hide(axis="index"))
caps = pd.Series(values["brand_value_by_cap_base_usd_m"]).rename("Literal brand value (US$m)").to_frame()
display(caps.style.format("{:,.0f}"))
md(f"**Tables 3–4.** Market capitalisation base: {values['market_cap_base'].replace('_', ' ')} = US${values['market_cap_usd_m']/1000:.1f}bn. "
   f"Block-bootstrap 90% range for the literal brand value: US${boot[0]:,.0f}m to US${boot[1]:,.0f}m (the interval includes zero and negative values). "
   "The Brand Finance row shows the share of market cap its published value implies; it is a different method and construct, shown for scale only.")

Reading,Brand share,Brand value (US$m)
Literal: share of all explained movement (owner-selected primary),2.10%,446
Lenovo-specific: share of the non-market drivers,23.76%,"5,039"
Lower bound: share of total return variance,0.38%,80
"Brand Finance 2025 (external, royalty-relief method)",26.70%,"5,663"


,Literal brand value (US$m)
pre_announcement_market_cap,366
latest_market_cap,"1,193"
mean_post_announcement_market_cap,446


**Tables 3–4.** Market capitalisation base: mean post announcement market cap = US$21.2bn. Block-bootstrap 90% range for the literal brand value: US$-364m to US$3,669m (the interval includes zero and negative values). The Brand Finance row shows the share of market cap its published value implies; it is a different method and construct, shown for scale only.

### 5.3 From the sponsorship's Index lift to incremental brand value

In [5]:
lit_inc, spec_inc = chain["incremental_brand_value_usd_m"], values["incremental_brand_value_lenovo_specific_usd_m"]
steps = pd.DataFrame([
    ("Total sponsorship effect Δ (Brand Index points)", f"{chain['sustained_lift_index_points']:+.2f}"),
    ("No-sponsorship level I_cf (mean synthetic index, post-announcement)", f"{chain['counterfactual_index_level']:.1f}"),
    ("Uplift u = Δ / I_cf (brand share of attention)", f"{chain['brand_share_uplift_pct']:+.1f}%"),
    ("Elasticity of brand value to brand share ε", f"{chain['brand_value_elasticity_to_brand_share']:.1f}"),
    ("Incremental brand value, literal reading (US$m)", f"{lit_inc:,.1f}"),
    ("  90% range from the brand-share bootstrap (US$m)", f"{chain['incremental_brand_value_90pct_usd_m'][0]:,.0f} to {chain['incremental_brand_value_90pct_usd_m'][1]:,.0f}"),
    ("  range from leave-one-donor-out lifts (US$m)", f"{chain['incremental_brand_value_donor_range_usd_m'][0]:,.1f} to {chain['incremental_brand_value_donor_range_usd_m'][1]:,.1f}"),
    ("  US$m per Index point", f"{chain['coefficient_usd_m_per_index_point']:,.2f}"),
    ("Incremental brand value, Lenovo-specific reading (US$m)", f"{spec_inc:,.0f}"),
    ("Incremental value at the Brand Finance reference (US$m, illustration)", f"{chain['brand_share_uplift_pct'] / 100 * ref['brand_finance_2025_brand_value_usd_m']:,.0f}"),
], columns=["Step", "Value"])
display(steps.style.hide(axis="index"))
md(f"**Table 5.** The sponsorship's lift of {chain['sustained_lift_index_points']:+.2f} points over {effect['total_effect']['post_weeks']} weeks "
   f"({effect['total_effect']['first_post_week']} to {effect['total_effect']['last_week']}) is a {chain['brand_share_uplift_pct']:+.1f}% uplift in brand share of attention. "
   "Incremental brand value is that percentage of brand value, under each reading.")

Step,Value
Total sponsorship effect Δ (Brand Index points),+7.48
"No-sponsorship level I_cf (mean synthetic index, post-announcement)",102.0
Uplift u = Δ / I_cf (brand share of attention),+7.3%
Elasticity of brand value to brand share ε,1.0
"Incremental brand value, literal reading (US$m)",32.7
90% range from the brand-share bootstrap (US$m),-27 to 269
range from leave-one-donor-out lifts (US$m),30.5 to 43.1
US$m per Index point,4.37
"Incremental brand value, Lenovo-specific reading (US$m)",370
"Incremental value at the Brand Finance reference (US$m, illustration)",415


**Table 5.** The sponsorship's lift of +7.48 points over 100 weeks (2024-10-21 to 2026-09-14) is a +7.3% uplift in brand share of attention. Incremental brand value is that percentage of brand value, under each reading.

### 5.4 Primary: incremental brand value from the FIFA-specific effect (ADR-0043)

In [6]:
u = fs["uplift_pct"]
tab = pd.DataFrame({reading: fs["incremental_brand_value_usd_m"][reading] for reading in ["literal", "lenovo_specific"]})
tab.insert(0, "Uplift (% brand share)", pd.Series(u))
tab = tab.rename(index={"primary": "Primary (FIFA-specific)", "statistical_band_95_low": "95% band, low", "statistical_band_95_high": "95% band, high",
                        "attribution_band_upper": "Attribution upper band (whole gap, same weeks)"},
                 columns={"literal": "Literal reading (US$m)", "lenovo_specific": "Lenovo-specific reading (US$m)"})
display(tab.style.format({"Uplift (% brand share)": "{:+.2f}%", "Literal reading (US$m)": "{:,.1f}", "Lenovo-specific reading (US$m)": "{:,.0f}"}))
md(f"**Table 6.** FIFA-specific effect {fs['index_points']['primary']:+.2f} Brand Index points ({fs['period'][0]} to {fs['period'][1]}), from `10_total_sponsorship_effect.ipynb` §5.8. "
   "Incremental brand value = brand value × uplift × elasticity (1). Table 5 above applies the total effect over the full post period, i.e. the upper attribution band.")

,Uplift (% brand share),Literal reading (US$m),Lenovo-specific reading (US$m)
Primary (FIFA-specific),+4.33%,19.3,218
"95% band, low",+1.67%,7.4,84
"95% band, high",+7.00%,31.2,353
"Attribution upper band (whole gap, same weeks)",+6.88%,30.7,347


**Table 6.** FIFA-specific effect +4.42 Brand Index points (2024-10-21 to 2026-07-20), from `10_total_sponsorship_effect.ipynb` §5.8. Incremental brand value = brand value × uplift × elasticity (1). Table 5 above applies the total effect over the full post period, i.e. the upper attribution band.

### 5.5 Comparison routes (market-implied, profit, royalty, event study, media value)

In [7]:
routes = pd.read_parquet(E + "valuation_routes_v1/valuation_routes.parquet")
display(routes[["route", "link_estimate", "link_lower_95", "link_upper_95", "link_identified", "value_unit", "value_at_measured_level"]].rename(columns={
    "route": "Route", "link_estimate": "Link estimate", "link_lower_95": "95% low", "link_upper_95": "95% high",
    "link_identified": "Link statistically identified", "value_unit": "Value unit", "value_at_measured_level": "Value at measured lift"})
    .style.format({"Link estimate": "{:.4f}", "95% low": "{:.4f}", "95% high": "{:.4f}", "Value at measured lift": "{:,.1f}"}, na_rep="—").hide(axis="index"))
be = pd.read_parquet(E + "valuation_routes_v1/breakeven.parquet")
royalty = be[be["route"].eq("relief_from_royalty")][["cost_usd_m", "required_royalty_uplift_basis_points"]]
display(royalty.rename(columns={"cost_usd_m": "Annual sponsorship cost (US$m)", "required_royalty_uplift_basis_points": "Royalty-rate uplift needed (bp)"})
        .style.format("{:.1f}").hide(axis="index"))
md(f"**Tables 7–8.** Routes with a statistically identified brand-to-money link: {', '.join(routes.loc[routes['link_identified'], 'route']) or 'none'}. "
   "The breakeven reframes the question: how much would the brand's royalty rate need to rise to cover a given annual cost. "
   "The sponsorship fee is not in the workspace, so costs are a grid.")

Route,Link estimate,95% low,95% high,Link statistically identified,Value unit,Value at measured lift
market_implied_equity_v3,0.0102,-0.0059,0.0263,False,"full-contract equity value, US$m","2,078.9"
market_implied_annual_earnings,0.0024,-0.0027,0.0076,False,"annual earnings-equivalent, US$m",37.3
direct_profit_margin,0.0090,-0.1100,0.1280,False,"annual adjusted net income, US$m (trailing-four-quarter revenue)",61.4
relief_from_royalty,—,—,—,False,"annual after-tax royalty saving per 1bp of royalty rate, US$m",—
announcement_event_study,0.0174,—,0.0425,False,"gross event equity value, US$m",306.2
industry_claimed_media_value,—,—,—,False,"media value 2025-05-01 to 2026-04-01, currency not stated",164.7


Annual sponsorship cost (US$m),Royalty-rate uplift needed (bp)
50.0,6.9
100.0,13.7
150.0,20.6
250.0,34.3


**Tables 7–8.** Routes with a statistically identified brand-to-money link: none. The breakeven reframes the question: how much would the brand's royalty rate need to rise to cover a given annual cost. The sponsorship fee is not in the workspace, so costs are a grid.

## 6. Interpretation

In [8]:
md(f"""- **Brand value:** under the owner-selected literal reading the brand is worth **US${values['brand_value_usd_m']:,.0f}m**
  ({100*values['brand_share_of_price_formation']:.2f}% of a US${values['market_cap_usd_m']/1000:.1f}bn market cap); on the Lenovo-specific reading
  **US${values['brand_value_lenovo_specific_usd_m']:,.0f}m** ({100*values['brand_share_of_lenovo_specific_formation']:.1f}%). For scale, Brand Finance publishes
  US${ref['brand_finance_2025_brand_value_usd_m']/1000:.2f}bn with a different method.
- **Incremental brand value from the FIFA sponsorship (primary):** the FIFA-specific uplift of {fs['uplift_pct']['primary']:+.1f}% gives
  **US${fs['incremental_brand_value_usd_m']['literal']['primary']:,.1f}m** on the literal reading (band US${fs['incremental_brand_value_usd_m']['literal']['statistical_band_95_low']:,.1f}m to
  US${fs['incremental_brand_value_usd_m']['literal']['attribution_band_upper']:,.1f}m) and **US${fs['incremental_brand_value_usd_m']['lenovo_specific']['primary']:,.0f}m** on the Lenovo-specific reading
  (band US${fs['incremental_brand_value_usd_m']['lenovo_specific']['statistical_band_95_low']:,.0f}m to US${fs['incremental_brand_value_usd_m']['lenovo_specific']['attribution_band_upper']:,.0f}m).
  The total effect over the full post period ({chain['brand_share_uplift_pct']:+.1f}%) gives US${lit_inc:,.1f}m / US${spec_inc:,.0f}m (upper attribution band).
- **Reference case:** Lenovo's brand value without the measured lift, at the mean post-announcement market capitalisation.
- **Decision relevance:** planning figures conditional on (i) the lift being real, which is borderline significant, and (ii) the brand's role in
  price formation, which share-price data measure only loosely (the bootstrap range includes zero). The two readings differ by a factor of
  {values['brand_value_lenovo_specific_usd_m'] / values['brand_value_usd_m']:.0f}; choosing between them is an owner decision, not a statistical one.""")

- **Brand value:** under the owner-selected literal reading the brand is worth **US$446m**
  (2.10% of a US$21.2bn market cap); on the Lenovo-specific reading
  **US$5,039m** (23.8%). For scale, Brand Finance publishes
  US$5.66bn with a different method.
- **Incremental brand value from the FIFA sponsorship (primary):** the FIFA-specific uplift of +4.3% gives
  **US$19.3m** on the literal reading (band US$7.4m to
  US$30.7m) and **US$218m** on the Lenovo-specific reading
  (band US$84m to US$347m).
  The total effect over the full post period (+7.3%) gives US$32.7m / US$370m (upper attribution band).
- **Reference case:** Lenovo's brand value without the measured lift, at the mean post-announcement market capitalisation.
- **Decision relevance:** planning figures conditional on (i) the lift being real, which is borderline significant, and (ii) the brand's role in
  price formation, which share-price data measure only loosely (the bootstrap range includes zero). The two readings differ by a factor of
  11; choosing between them is an owner decision, not a statistical one.

## 7. Limitations and non-claims

- **Not an accounting or audited value**, and not a causal estimate. The share-price relative-importance proxy stands in for the behavioural evidence on demand that ISO 10668 expects.
- **Variance share ≠ value share.** A stable brand explains little short-run price movement yet can be worth a lot; this biases the dominance-based brand share downward, most strongly in the literal reading.
- **FIFA-specific split is a regression decomposition** (`10_total_sponsorship_effect.ipynb` §5.8), with its specification fixed by the owner after it was seen (ADR-0043).
- **Lift is borderline.** The incremental value inherits the total effect's borderline significance and the caveats in `10_total_sponsorship_effect.ipynb`, including that the Brand Index was redesigned after the former headline was known (ADR-0039/0040).
- **Assumed elasticity.** Brand value is assumed proportional to brand share of attention ($\varepsilon = 1$).
- **Point-in-time prices.** Market capitalisation bases are observed; brand value scales with them.
- **DCF retired.** The economic-profit DCF (ADR-0033) is no longer computed (owner decision, ADR-0040); its figures in earlier ADRs are historical.
- **External reference is not validation.** Brand Finance uses royalty relief and its own brand-strength scoring.

## Next steps

1. Add a purchase-drivers question set to the post-World Cup survey to estimate the brand's role from demand rather than share-price movement.
2. Refresh after each data pull; the total effect will include the full World Cup persistence window after 18 October 2026.
3. Obtain the sponsorship fee and activation cost to turn value into a return on investment.